# Speculative Decoding

[【手撕LLM-Speculative Decoding】大模型迈向"并行"解码时代](https://zhuanlan.zhihu.com/p/671432448)

1. 目标模型
2. 草稿模型

# Config

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)

In [31]:
# Config

# model
dim = 128
num_layers = 2
vocab_size = 100

# data
batch_size = 1
seq_len = 200

# speculative hyper parameters
spec_n = 5

# Model

In [32]:
class DecoderBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, dim) # attn shoould with kvcache
        self.ffn = nn.Linear(dim, dim)
        self.norm_1 = nn.Linear(dim, dim)
        self.norm_2 = nn.Linear(dim, dim)
        self.gelu = nn.GELU()
    def forward(self, X):
        X = self.attn( self.norm_1(X)) + X
        X = self.gelu(self.ffn( self.norm_2(X))) + X
        return X
        
class LanguageModel(nn.Module):
    def __init__(self, dim, vocab_size, num_layers):
        super().__init__()
        self.embd = nn.Embedding(vocab_size, dim)
        self.decoder = nn.ModuleList(
            DecoderBlock(dim) for _ in range(num_layers)
        )
        self.norm = nn.Linear(dim, dim)
        self.lm_heads = nn.Linear(dim, vocab_size)
    def forward(self, x):
        X = self.embd(x)
        for block in self.decoder:
            X = block(X)
        H = self.norm(X)
        logits = self.lm_heads(H)
        return logits

model = LanguageModel(dim, vocab_size, num_layers)
model.eval()
x = torch.randint(vocab_size, (batch_size, seq_len))
logits = model(x)
print(x.shape, logits.shape)

torch.Size([1, 200]) torch.Size([1, 200, 100])


## draft model

In [33]:
from copy import deepcopy
model_draft = deepcopy((model))
model_draft.eval()
model_draft.lm_heads.weight.data[:, :] *= (torch.randn(vocab_size, dim)+1.0) # noise
print(model.lm_heads.weight)
print(model_draft.lm_heads.weight)
logits_ = model_draft(x)

Parameter containing:
tensor([[ 0.0518,  0.0407,  0.0631,  ...,  0.0408, -0.0484,  0.0876],
        [-0.0094,  0.0788, -0.0725,  ...,  0.0340, -0.0564,  0.0403],
        [ 0.0698, -0.0671,  0.0542,  ...,  0.0512,  0.0148,  0.0596],
        ...,
        [-0.0194,  0.0340, -0.0648,  ..., -0.0675,  0.0601,  0.0704],
        [-0.0351, -0.0692,  0.0045,  ...,  0.0058,  0.0579, -0.0476],
        [ 0.0686, -0.0261,  0.0387,  ..., -0.0610,  0.0340,  0.0790]],
       requires_grad=True)
Parameter containing:
tensor([[-2.3393e-04, -7.8469e-02,  2.2370e-02,  ...,  1.8888e-02,
         -8.8269e-02,  2.4296e-01],
        [-2.0078e-02, -1.7656e-02, -1.2742e-01,  ..., -1.1659e-02,
         -5.4449e-02,  6.8560e-02],
        [-1.7589e-02, -1.1529e-01,  1.5552e-01,  ..., -1.2817e-02,
          4.5334e-02,  8.3142e-03],
        ...,
        [-2.2857e-02,  4.9903e-02, -9.3997e-02,  ..., -6.6474e-02,
         -3.6422e-02,  1.1312e-01],
        [-1.8928e-02, -5.7012e-02,  4.9333e-03,  ...,  1.1223e-02,
   

# KL

度量两者输出分布

In [34]:
def KL(logits_p, logits_q):
    
    log_p = F.log_softmax(logits_p, dim=-1)
    q = F.softmax(logits_q, dim=-1)
    kl = F.kl_div(log_p, q, reduction='sum', log_target=False)
    
    return kl

# target-draft
result = KL(logits, logits_)
print(result)

# target-random
logits_ = torch.randn(batch_size,seq_len, vocab_size)
result = KL(logits, logits_)
print(result)

# target-target
result = KL(logits, logits_)
print(result)

tensor(14.7047, grad_fn=<SumBackward0>)
tensor(110.3888, grad_fn=<SumBackward0>)
tensor(110.3888, grad_fn=<SumBackward0>)


# Speculative Decoding

In [35]:
x_draft = torch.tensor([23, 19, 30, 62, 70])
y_draft = torch.tensor([[19, 30, 62, 70, 20]], dtype=torch.long)
y_target = torch.tensor([[19, 30, 62, 10, 30]], dtype=torch.long)

# 正常情况
verify = y_draft == y_target
print(verify)

idx_1, idx2 = torch.where(verify==False)
print(idx2[0])

# 全接受
y_target = torch.tensor([[19, 30, 62, 70, 20]], dtype=torch.long)
verify = y_draft == y_target
print(verify)

idx_1, idx2 = torch.where(verify==False)
print(len(idx2))
# print(idx2[0])

# 无接受
y_target = torch.tensor([[1, 2, 3, 4, 5]], dtype=torch.long)
verify = y_draft == y_target
print(verify)

idx_1, idx2 = torch.where(verify==False)
print(len(idx2))
print(idx2[0])

tensor([[ True,  True,  True, False, False]])
tensor(3)
tensor([[True, True, True, True, True]])
0
tensor([[False, False, False, False, False]])
5
tensor(0)


## Basic SP

In [36]:
a = torch.randn(2,20)

torch.argmax(a, dim=-1, keepdim=True).shape

torch.Size([2, 1])

In [37]:
class SPDecoding:
    def __init__(self, model_target, model_draft, spec_n):
        self.model_target = model_target
        self.model_draft = model_draft
        self.spec_n = spec_n

    def generate_draft(self, spec_n, x):
        logits_y = []
        for i in range(spec_n):
            with torch.no_grad():
                logits = self.model_draft(x)[:, [-1], :]
                logits_y.append(logits)
                
                # argmax
                next_token = torch.argmax(logits, dim = -1) # [bsz, 1]
                x = torch.cat([x, next_token], dim=1)
        return x, torch.cat(logits_y, dim=1)
        
    def generate(self, x, max_new_tokens=30):
        # x[bsz, seq_len], 仅考虑 bsz=1
        # 此版本实现不考虑 KV Cache
        count = 0 # 生成数量
        y_new = []
        for i in range(max_new_tokens):
            # 猜测
            x_spec, logits_draft = self.generate_draft(self.spec_n, x)
            y_spec = x_spec[:, -self.spec_n:]
            
            logits_target = self.model_target(x_spec)[:, -self.spec_n-1:]
            y_target = torch.argmax(logits_target, dim = -1)

            # 验证
            y_target_verify = y_target[:, :-1] # y_target 比 y_spec 多预测一个 token
            # print(y_target, y_spec)
            verify = y_spec == y_target_verify
            idx1, idx2 = torch.where(verify==False)
            if len(idx2) == 0:
                # 全接受, y_target 得到 6 个 token
                accept_len = self.spec_n
            else:
                # 部分接受
                accept_len = idx2[0]

            # 更新
            print(f'step:{i}, accept_len:{accept_len}')
            x = torch.cat((x, y_target[:, :accept_len+1]), dim = 1)
            print('accep token',y_target[:,:accept_len+1])
            y_new.append(y_target[:,:accept_len+1])
            # 注意: 验证成功 1 个token, 本身 target_model 基于 1 个正确的 token, 验证出来的 token 也认为是对的
            
            # 更新 kvcache
            # ...

            count += (accept_len+1)
            if count >= max_new_tokens-1:
                print(f'speedup: {max_new_tokens/i}x') # sp step / max_new_tokens
                break

        return torch.cat(y_new, dim = 1), max_new_tokens/i

In [38]:
sp = SPDecoding(model_target=model, model_draft=model_draft, spec_n=spec_n)

In [39]:
y_new, ratio = sp.generate(x, max_new_tokens=20)
print(y_new)
print(ratio)

step:0, accept_len:0
accep token tensor([[76]])
step:1, accept_len:0
accep token tensor([[43]])
step:2, accept_len:0
accep token tensor([[27]])
step:3, accept_len:0
accep token tensor([[73]])
step:4, accept_len:0
accep token tensor([[61]])
step:5, accept_len:0
accep token tensor([[43]])
step:6, accept_len:0
accep token tensor([[27]])
step:7, accept_len:0
accep token tensor([[73]])
step:8, accept_len:0
accep token tensor([[61]])
step:9, accept_len:0
accep token tensor([[43]])
step:10, accept_len:0
accep token tensor([[27]])
step:11, accept_len:0
accep token tensor([[73]])
step:12, accept_len:0
accep token tensor([[61]])
step:13, accept_len:0
accep token tensor([[43]])
step:14, accept_len:0
accep token tensor([[27]])
step:15, accept_len:0
accep token tensor([[73]])
step:16, accept_len:0
accep token tensor([[61]])
step:17, accept_len:0
accep token tensor([[43]])
step:18, accept_len:0
accep token tensor([[27]])
speedup: 1.1111111111111112x
tensor([[76, 43, 27, 73, 61, 43, 27, 73, 61, 43, 2

## Basic Speculative Decoding 讨论

解码数量

1. 输入长度为 L 的 token, 给猜测模型解码 5 个token, 此时有 L+5 token
2. 目标模型对 (L+5) 个token进行前向, 最终可以得到 (L+5) 个 logits, 而最后一个 logits, 可以采样出第 L+5+1 个 token
3. 此时 y_spec 摘取 5 token, y_target 摘取 6 个 token,  y_target 检验前 5 个 token

接收讨论
1. 如果未有接受（退化为标准的解码）, y_target 最后一个 token 视为标准的 next_token_prediction
2. 部分接受, 如猜测解码2个被接受, 此时有 y_target 有 2 个可用，加上本身做next token，最终 y_target 有 3 个可用
3. 如果全接受, 接受 6 个token

KVCache 讨论

1. generate 循环前做一次 prefill
2. 往后, 仅输入“解码序列”, 对于目标模型来说, 其模式为输入: `y_target[:, :accept_len+1]` 而非 decoding-stage 的单 token

# Speculative Sampling

上述草稿模型通过 greedy-sampling , 生成猜测解码。 如果是生成是随机采样呢? 

1. 猜测时是随机采样, 解码时也是随机采样
2. 如果接受了 2 个 token, target-model 可以基于这两个 token 做 next token 呢？
3. 论文给出

![](sp-sampling.png)

In [40]:
A = torch.randn(2,3)
prob = F.softmax(A, dim = -1)
torch.multinomial(prob, num_samples=1)

tensor([[1],
        [0]])

In [43]:
class SPSamplingDecoding:
    def __init__(self, model_target, model_draft, spec_n):
        self.model_target = model_target
        self.model_draft = model_draft
        self.spec_n = spec_n

    def generate_draft(self, spec_n, x):
        logits_y = []
        for i in range(spec_n):
            with torch.no_grad():
                logits = self.model_draft(x)[:, [-1], :]
                logits_y.append(logits)
                
                # + sampling
                prob = F.softmax(logits, dim=-1)
                next_token = self._sampling(prob)
                
                x = torch.cat([x, next_token], dim=1)
        return x, torch.cat(logits_y, dim=1)

    def _sampling(self, prob):
        B, L, V = prob.shape
        next_token = torch.multinomial(prob.reshape(B*L, V), 1)
        next_token = next_token.reshape(B, L)
        return next_token
        
    def generate(self, x, max_new_tokens=30):
        # x[bsz, seq_len], 仅考虑 bsz=1
        # 此版本实现不考虑 KV Cache
        count = 0 # 生成数量
        y_new = []
        for i in range(max_new_tokens):
            # 猜测
            x_spec, logits_draft = self.generate_draft(self.spec_n, x)
            y_spec = x_spec[:, -self.spec_n:]
            
            logits_target = self.model_target(x_spec)[:, -self.spec_n-1:]
            # + sampling
            prob = F.softmax(logits_target, dim=-1)
            y_target = self._sampling(prob)

            # Speculative Decoding
            accept_len = 0
            next_tokens = []
            for j in range(self.spec_n):
                # sample random from uniform distribution
                r = torch.rand(1).item()

                token_id = y_spec[0, j]
                q = F.softmax(logits_target[0, j], dim = -1)
                p = F.softmax(logits_draft[0, j], dim = -1)

                # print(q[token_id],p[token_id])
                if r < min(1, q[token_id]/p[token_id]):
                    accept_len += 1
                    next_tokens.append(y_spec[0, j])
                else:
                    # re-sampling

                    q_ = q.clone()
                    idx = torch.where( q < p )
                    
                    q_[idx] = p[idx]


                    next_token = torch.multinomial(q_, num_samples=1)
                    next_tokens.append(next_token)
                    break
                    
            if accept_len == self.spec_n:
                next_tokens.append(y_target[0, accept_len])
                accept_len+=1

            # 更新输入
            next_tokens = torch.tensor([next_tokens], dtype=torch.long)
            x = torch.cat((x, next_tokens), dim = -1)
            y_new.append(next_tokens)
            print(f'step {i}: accept_len: {accept_len}')

            
            # 更新 kvcache
            # ...
            count+=accept_len
            if count >= max_new_tokens-1:
                print(f'speedup: {max_new_tokens/i}x') # sp step / max_new_tokens
                break

        return torch.cat(y_new, dim = 1), max_new_tokens/i

In [44]:
sp = SPSamplingDecoding(model_target=model, model_draft=model_draft, spec_n=spec_n)
y_new, ratio = sp.generate(x, max_new_tokens=20)
print(y_new)
print(ratio)

step 0: accept_len: 6
step 1: accept_len: 1
step 2: accept_len: 6
step 3: accept_len: 6
speedup: 6.666666666666667x
tensor([[71, 63, 34, 93, 70, 51, 19, 28, 26, 54, 26, 73, 33, 36, 94,  8, 53, 86,
         67, 83]])
6.666666666666667
